# Distribution Shift, Monitoring, and Governance Lab


## Purpose

This lab simulates distribution shift between training and deployment data.

Learning goals:

- Compare reference and current feature distributions.
- Compute a simple population stability index.
- Interpret drift as a monitoring signal, not a complete governance process.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

reference = rng.normal(loc=0.0, scale=1.0, size=5000)
current_mild = rng.normal(loc=0.35, scale=1.05, size=5000)
current_severe = rng.normal(loc=1.00, scale=1.25, size=5000)

def population_stability_index(reference_values, current_values, bins=10):
    reference_counts, bin_edges = np.histogram(reference_values, bins=bins)
    current_counts, _ = np.histogram(current_values, bins=bin_edges)

    reference_pct = reference_counts / max(reference_counts.sum(), 1)
    current_pct = current_counts / max(current_counts.sum(), 1)

    epsilon = 1e-6
    return np.sum(
        (current_pct - reference_pct)
        * np.log((current_pct + epsilon) / (reference_pct + epsilon))
    )

drift = pd.DataFrame([
    {
        "comparison": "reference_vs_mild_shift",
        "psi": population_stability_index(reference, current_mild),
    },
    {
        "comparison": "reference_vs_severe_shift",
        "psi": population_stability_index(reference, current_severe),
    },
])

drift

## Governance Questions

- What threshold triggers review?
- Who receives the alert?
- Is retraining automatic or governed?
- What data is approved for retraining?
- How are model versions and rollback plans documented?